# CYR-GPU-004 - LR-retention test using REAL Cymek V5

**Runtime: T4 GPU (NOT TPU, NOT CPU)**

Runtime -> Change runtime type -> **T4 GPU** -> Run all -> ~2-3 hours -> auto-download


In [ ]:
# CELL 0: SETUP
import subprocess, sys
subprocess.run(['git', 'clone', '--branch', 'cymek-500m-readiness',
    'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',
    '/content/repo'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers'], check=True)
import os; os.chdir('/content/repo')
import torch
assert torch.cuda.is_available(), 'CYR-GPU-004 requires GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)


In [ ]:
# CELL 1: RUN EXPERIMENT (4 arms: 2 seeds x 2 LR)
import subprocess, sys
for seed, lr, label in [(707, 0.001, 'HIGH'), (707, 1e-05, 'LOW'),
                         (808, 0.001, 'HIGH'), (808, 1e-05, 'LOW')]:
    out = 'experiments/ARK-007/CYR_GPU_004_seed{}_lr{}_RESULT.json'.format(seed, lr)
    print('\n=== seed {} lr {} ({}) ==='.format(seed, lr, label), flush=True)
    subprocess.run([sys.executable, '-u', 'experiments/ARK-007/run_v4.py',
        '--seed', str(seed), '--lr', str(lr),
        '--acq-steps', '16000', '--post-steps', '8000',
        '--device', 'cuda', '--out', out], check=True)
print('\nALL ARMS COMPLETE')


In [ ]:
# CELL 2: SUMMARY + DOWNLOAD
import json, glob, os
all_r = {}
for f in sorted(glob.glob('experiments/ARK-007/CYR_GPU_004_*RESULT*.json')):
    r = json.load(open(f, encoding='utf-8'))
    all_r[os.path.basename(f)] = r
print(json.dumps(all_r, indent=1, default=str))
try:
    from google.colab import files
    import shutil
    shutil.make_archive('/content/CYR-GPU-004_RESULTS', 'zip', 'experiments/ARK-007')
    files.download('/content/CYR-GPU-004_RESULTS.zip')
except Exception as exc:
    print('manual download from experiments/ARK-007/:', exc)
